In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_sales_df = spark.table(
    "workspace.default.bronze_raw_sales"
)

bronze_products_df = spark.table(
    "workspace.default.bronze_raw_products"
)

print(f"Bronze sales rows: {bronze_sales_df.count():,}")
print(f"Bronze product rows: {bronze_products_df.count():,}")

In [0]:
silver_sales_df = (
    bronze_sales_df
    .select(
        F.col("sales_id").cast("long").alias("sales_id"),
        F.col("product_id").cast("long").alias("product_id"),
        F.col("cust_id").cast("long").alias("customer_id"),
        F.col("qty").cast("integer").alias("quantity"),
        F.col("unit_price").cast("decimal(12, 2)").alias("unit_price"),
        F.col("total_amt").cast("decimal(14, 2)").alias("total_amount"),
        F.to_timestamp("sale_date").alias("ordered_at"),
        F.col("source_system"),
        F.col("ingested_at")
    )
    .filter(F.col("sales_id").isNotNull())
    .filter(F.col("product_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("quantity") > 0)
    .filter(F.col("unit_price") >= 0)
    .dropDuplicates(["sales_id", "product_id"])
    .withColumn(
        "order_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("sales_id").cast("string"),
                F.col("product_id").cast("string")
            ),
            256
        )
    )
    .withColumn(
        "calculated_total",
        F.round(F.col("quantity") * F.col("unit_price"), 2)
    )
    .withColumn(
        "revenue_is_valid",
        F.abs(F.col("total_amount") - F.col("calculated_total"))
        <= F.lit(0.01)
    )
)

display(silver_sales_df.limit(10))

In [0]:
product_window = (
    Window
    .partitionBy("id")
    .orderBy(F.col("ingested_at").desc())
)

silver_products_df = (
    bronze_products_df
    .filter(F.col("id").isNotNull())
    .withColumn(
        "record_number",
        F.row_number().over(product_window)
    )
    .filter(F.col("record_number") == 1)
    .select(
        F.col("id").cast("long").alias("product_id"),
        F.upper(F.trim(F.col("title"))).alias("product_name"),
        F.trim(F.col("description")).alias("product_description"),
        F.col("price").cast("decimal(12, 2)").alias("price"),
        F.trim(F.col("category")).alias("category"),
        F.col("image").alias("image_url"),
        F.regexp_extract(
            F.col("rating"),
            r"'rate':\s*([0-9.]+)",
            1
        ).cast("double").alias("rating"),
        F.regexp_extract(
            F.col("rating"),
            r"'count':\s*(\d+)",
            1
        ).cast("integer").alias("rating_count"),
        F.col("source_system"),
        F.col("ingested_at")
    )
    .withColumn(
        "price_level",
        F.when(F.col("price") < 20, "Low")
        .when(F.col("price") < 100, "Medium")
        .otherwise("High")
    )
)

display(silver_products_df)

In [0]:
(
    silver_sales_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.silver_sales")
)

(
    silver_products_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.silver_products")
)

print("Created workspace.default.silver_sales")
print("Created workspace.default.silver_products")

In [0]:
sales_validation_df = spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT sales_id) AS distinct_sales,
        SUM(CASE WHEN revenue_is_valid = false THEN 1 ELSE 0 END)
            AS invalid_revenue_rows,
        SUM(CASE WHEN ordered_at IS NULL THEN 1 ELSE 0 END)
            AS invalid_timestamp_rows
    FROM workspace.default.silver_sales
""")

product_validation_df = spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT product_id) AS distinct_products,
        SUM(CASE WHEN product_name IS NULL THEN 1 ELSE 0 END)
            AS missing_product_names,
        SUM(CASE WHEN price < 0 THEN 1 ELSE 0 END)
            AS invalid_prices
    FROM workspace.default.silver_products
""")

display(sales_validation_df)
display(product_validation_df)